In [2]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# 1. Setup Models
# Make sure your database was built with this SAME embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
persistent_directory = "db1/chroma_db"

db = Chroma(persist_directory=persistent_directory, embedding_function=embedding_model)


model = ChatGroq(
    model_name="llama-3.3-70b-versatile", 
    temperature=0,
)

# 2. Conversation History
chat_history = []

def ask_question(user_question):
    global chat_history
    print(f"\n--- Processing: {user_question} ---")
    
    # STEP 1: Contextualize the question
    # This turns "How much did they pay?" into "How much did Microsoft pay for GitHub?"
    search_question = user_question
    if chat_history:
        context_prompt = [
            SystemMessage(content="Given the chat history and a new question, rewrite it as a standalone question that can be understood without the history. Just return the text of the new question."),
        ] + chat_history + [HumanMessage(content=f"Rewrite this question: {user_question}")]
        
        result = model.invoke(context_prompt)
        search_question = result.content.strip()
        print(f"Standalone Search Query: {search_question}")

    # STEP 2: Retrieve Documents
    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_question)
    
    # STEP 3: Generate Answer with Context
    context_text = "\n".join([f"- {doc.page_content}" for doc in docs])
    combined_input = f"""Answer the question using ONLY the provided documents.
    
    Documents:
    {context_text}
    
    Question: {user_question}
    """

    messages = [
        SystemMessage(content="You are a helpful assistant. Use the provided context to answer questions accurately."),
    ] + chat_history + [HumanMessage(content=combined_input)]
    
    response = model.invoke(messages)
    answer = response.content
    
    # STEP 4: Update History
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))
    
    # Keep history manageable (last 6 messages)
    if len(chat_history) > 10:
        chat_history = chat_history[-10:]
        
    print(f"\nAnswer: {answer}")

def start_chat():
    print("Welcome to your RAG Chat! (Type 'quit' to exit)")
    while True:
        user_input = input("\nYour question: ")
        if user_input.lower() == 'quit':
            break
        ask_question(user_input)

if __name__ == "__main__":
    start_chat()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2500.21it/s]


Welcome to your RAG Chat! (Type 'quit' to exit)

--- Processing: In what year did Tesla begin production of the Roadster? ---

Answer: According to the provided documents, Tesla began production of the Roadster in 2008.

--- Processing: so after that what will they do ---
Standalone Search Query: What will happen next or what actions will be taken after that point?

Answer: There is no information provided to answer the question "so after that what will they do" as it is unclear what "that" refers to. The provided documents are a collection of news articles about various topics, including Tesla, Microsoft, and the White House, but they do not provide a clear context for the question.

--- Processing: what is the networth ---
Standalone Search Query: What is the current net worth?

Answer: The documents provided do not mention the net worth of an individual, but rather the valuation of SpaceX, a company founded by Elon Musk. 

According to the documents, the valuation of SpaceX is:
- $4

In [ ]:
# Synthetic Questions: 

# 1. "What was NVIDIA's first graphics accelerator called?"
# 2. "Which company did NVIDIA acquire to enter the mobile processor market?"
# 3. "What was Microsoft's first hardware product release?"
# 4. "How much did Microsoft pay to acquire GitHub?"
# 5. "In what year did Tesla begin production of the Roadster?"
# 6. "Who succeeded Ze'ev Drori as CEO in October 2008?"
# 7. "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?"
# 8. "What was the original name of Microsoft before it became Microsoft?"